# Data Center Environmental Justice — Pattern Analysis

This notebook:
1. Loads scraped data center locations
2. Overlays demographic data from the US Census
3. Overlays water stress data from WRI Aqueduct
4. Generates an interactive map for community use
5. Flags data gaps as findings

**Geographic scope:** Virginia and Texas (Phase 1)

In [ ]:
import pandas as pd
import folium
import requests
import json
import os
from folium.plugins import MarkerCluster

print('Libraries loaded.')

## 1. Load Scraped Data

In [ ]:
# Load enriched data center data
df = pd.read_csv('../data/datacenter_aq_enriched.csv')
print(f'Loaded {len(df)} data centers')
df.head()

In [ ]:
# Basic stats
print('Data centers by region:')
print(df['region'].value_counts())

print('\nData gaps (missing PM2.5 readings):')
missing = df['aq_pm25_last'].isna().sum()
print(f'  {missing} of {len(df)} locations ({missing/len(df)*100:.1f}%) have no nearby AQ station')
print('  → These gaps may reflect undermonitoring in affected communities.')

## 2. Fetch Demographic Data (US Census)

We pull median household income by county from the ACS 5-year estimates.
No API key required for basic queries.

In [ ]:
def get_census_income(state_fips):
    """
    Fetch median household income by county from Census ACS.
    state_fips: '51' = Virginia, '48' = Texas
    """
    url = (
        f'https://api.census.gov/data/2022/acs/acs5'
        f'?get=NAME,B19013_001E'
        f'&for=county:*'
        f'&in=state:{state_fips}'
    )
    resp = requests.get(url)
    data = resp.json()
    cols = data[0]
    rows = data[1:]
    return pd.DataFrame(rows, columns=cols).rename(
        columns={'B19013_001E': 'median_income', 'NAME': 'county_name'}
    )

va_income = get_census_income('51')  # Virginia
tx_income = get_census_income('48')  # Texas
census_df = pd.concat([va_income, tx_income])
census_df['median_income'] = pd.to_numeric(census_df['median_income'], errors='coerce')
print(f'Loaded income data for {len(census_df)} counties')
census_df.head()

## 3. Build Interactive Map

In [ ]:
# Centre map on US East
m = folium.Map(location=[35.0, -90.0], zoom_start=5, tiles='CartoDB positron')

cluster = MarkerCluster(name='Data Centers').add_to(m)

for _, row in df.dropna(subset=['lat', 'lon']).iterrows():
    # Colour by PM2.5 level
    pm25 = row.get('aq_pm25_last')
    if pd.isna(pm25):
        color = 'gray'     # no data — flag as monitoring gap
    elif pm25 > 12:
        color = 'red'      # above EPA annual standard
    elif pm25 > 8:
        color = 'orange'
    else:
        color = 'green'

    popup_text = f"""
    <b>{row.get('name', 'Unnamed Facility')}</b><br>
    Region: {row.get('region', '')}<br>
    Operator: {row.get('operator', 'Unknown')}<br>
    Nearest AQ Station: {row.get('aq_station', 'None found')}<br>
    PM2.5: {pm25 if pd.notna(pm25) else '⚠️ No monitoring data'} μg/m³<br>
    <i>Gray markers = no nearby air quality monitoring station.<br>
    This may reflect undermonitoring, not absence of pollution.</i>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=8,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=row.get('name', 'Data Center')
    ).add_to(cluster)

# Legend
legend_html = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
            background:white; padding:12px; border-radius:8px;
            border:1px solid #ccc; font-size:13px; font-family:sans-serif;">
  <b>PM2.5 Air Quality</b><br>
  <span style="color:green">●</span> Good (&lt;8 μg/m³)<br>
  <span style="color:orange">●</span> Moderate (8–12 μg/m³)<br>
  <span style="color:red">●</span> Above EPA standard (&gt;12 μg/m³)<br>
  <span style="color:gray">●</span> No monitoring data (data gap)<br>
  <br><small>Source: OpenAQ · OpenStreetMap · Census ACS</small>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)

os.makedirs('../visualizations', exist_ok=True)
m.save('../visualizations/map.html')
print('Map saved → visualizations/map.html')
m

## 4. Pattern Analysis — Do Data Centers Cluster Near Lower-Income Areas?

In [ ]:
# This is a simplified proximity check.
# Phase 2 will implement proper spatial join with county geometries.

# For now: flag Virginia counties with data centers
# and compare income distribution against statewide median

va_median = census_df[census_df['state'] == '51']['median_income'].median()
tx_median = census_df[census_df['state'] == '48']['median_income'].median()

print(f'Virginia statewide median household income: ${va_median:,.0f}')
print(f'Texas statewide median household income: ${tx_median:,.0f}')
print()
print('NOTE: Proper spatial join (data centers → counties → income) coming in Phase 2.')
print('This requires county geometry files (Census TIGER shapefiles).')

## 5. Data Gap Audit

Treating missing data as a finding — not a limitation.

In [ ]:
gap_report = {
    'total_facilities': len(df),
    'missing_name': df['name'].isna().sum() + (df['name'] == 'Unknown').sum(),
    'missing_operator': (df['operator'] == '').sum() + df['operator'].isna().sum(),
    'missing_aq_data': df['aq_pm25_last'].isna().sum(),
    'notes': {
        'missing_name': 'Unnamed facilities — operator not publicly disclosed',
        'missing_operator': 'No operator recorded in public OSM data',
        'missing_aq_data': 'No AQ monitoring within 25km — potential undermonitoring in affected communities'
    }
}

print(json.dumps(gap_report, indent=2))

# Save gap report
with open('../data/gap_report.json', 'w') as f:
    json.dump(gap_report, f, indent=2)
print('\nGap report saved → data/gap_report.json')

---
**Next steps (Phase 2):**
- Spatial join with Census TIGER county geometries for proper income overlay
- Add WRI Aqueduct water stress layer
- Add community annotation mechanism
- Integrate with Data Center Impact Dashboard pipeline